<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# Design Probes for the E6, E9 and E10 Plans

Probes, not tests. The plans in `docs/hypotheses/experiments/E6-E10.md` fix three small models before anything is
scored; E4's plan was parked because its network, written without a pilot, could not show what its hypothesis
changes. These probes check, on three seeds (1001 to 1003) that the plans' tests never use (their seeds are 1 to 20),
that each model is not decided by construction or degenerate - that the condition without the mechanism under test
does show the failure the mechanism is meant to prevent, and that the mechanism can change it. They informed the
choices each plan names; they decide nothing about any hunch. Three seeds give no interval, and none is claimed.

Run with `jupyter nbconvert --to notebook --execute --inplace notebooks/e6_e9_e10_design_probes.ipynb`; numpy as
pinned in `notebooks/requirements.txt`.

## E6: separation before completion

A store of M = 1000 units with k = 50 active, written by the covariance rule. 50 patterns, either independent
(*random*) or in ten families of five whose members each keep `keep` of their family prototype's 50 units (*similar*).
With separation, every pattern and cue first passes through a fixed sparse random projection (connection probability
0.05, Gaussian weights) and k-winners-take-all (ties to the lower index). A cue moves 5 of a pattern's 50 active units.
Retrieval swaps one pair at a time, which keeps 50 units active: the active unit with the lowest input (the lower index
among ties) and the inactive unit with the highest input (the lower index among ties) swap when the inactive one's
input, less the weight between the two, exceeds the active one's. Each swap lowers the network's energy, so the state
cannot cycle; it has settled when that pair does not swap, and is stopped after 5000 swaps. A first version updated
all units at once and left many cues in two-state cycles; the review of the plan found it, and this one replaced it.
Success: the settled state overlaps the target's stored code by at least 0.9 and more than any other stored code.
Printed per run: success share, unsettled share. The question for the plan: is there a family overlap at which the
store without separation interferes but not completely?

In [1]:
import numpy as np

M, K = 1000, 50
F = K / M

def kwta(h, k=K):
    s = np.zeros(h.shape[0])
    s[np.argsort(-h, kind="stable")[:k]] = 1.0
    return s

def patterns(rng, cond, keep, n=50, fam=5):
    out = []
    if cond == "random":
        for _ in range(n):
            s = np.zeros(M); s[rng.choice(M, K, replace=False)] = 1; out.append(s)
    else:
        for _ in range(n // fam):
            proto = rng.choice(M, K, replace=False)
            for _ in range(fam):
                kept = rng.choice(proto, keep, replace=False)
                add = rng.choice(np.setdiff1d(np.arange(M), kept), K - keep, replace=False)
                s = np.zeros(M); s[np.concatenate([kept, add])] = 1; out.append(s)
    return np.array(out)

def settle(W, s, limit=5000):
    h = W @ s
    for _ in range(limit):
        on, off = np.flatnonzero(s > 0), np.flatnonzero(s == 0)
        i, j = on[np.argmin(h[on])], off[np.argmax(h[off])]
        if h[j] - W[i, j] <= h[i]:
            return s, True
        s[i], s[j] = 0.0, 1.0
        h += W[:, j] - W[:, i]
    return s, False

def e6(seed, cond, sep, keep, moved=5, cues=100):
    rng = np.random.default_rng(seed)
    X = patterns(rng, cond, keep)
    R = rng.standard_normal((M, M)) * (rng.random((M, M)) < 0.05)
    enc = (lambda x: kwta(R @ x)) if sep else (lambda x: x.copy())
    C = np.array([enc(x) for x in X])
    W = (C - F).T @ (C - F)
    np.fill_diagonal(W, 0)
    ok = unsettled = 0
    for _ in range(cues):
        mu = rng.integers(len(X)); x = X[mu].copy()
        x[rng.choice(np.flatnonzero(x), moved, replace=False)] = 0
        x[rng.choice(np.flatnonzero(X[mu] == 0), moved, replace=False)] = 1
        s, settled = settle(W, enc(x))
        if not settled:
            unsettled += 1
            continue
        ov = C @ s / K
        if ov[mu] >= 0.9 and (ov >= ov[mu]).sum() == 1:
            ok += 1
    return ok / cues, unsettled / cues

print("keep seed | random: off on | similar: off on   (success share, unsettled share)")
for keep in (20, 25, 30, 35):
    for seed in (1001, 1002, 1003):
        r = {(c, s): e6(seed, c, s, keep) for c in ("random", "similar") for s in (0, 1)}
        print(keep, seed, "|", r[("random", 0)], r[("random", 1)], "|", r[("similar", 0)], r[("similar", 1)])


keep seed | random: off on | similar: off on   (success share, unsettled share)


20 1001 | (1.0, 0.0) (1.0, 0.0) | (1.0, 0.0) (1.0, 0.0)


20 1002 | (1.0, 0.0) (1.0, 0.0) | (1.0, 0.0) (1.0, 0.0)


20 1003 | (1.0, 0.0) (1.0, 0.0) | (1.0, 0.0) (1.0, 0.0)


25 1001 | (1.0, 0.0) (1.0, 0.0) | (0.6, 0.0) (1.0, 0.0)


...

30 1002 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.0) (0.95, 0.0)


30 1003 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.0) (0.82, 0.0)


35 1001 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.0) (0.04, 0.0)


35 1002 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.0) (0.06, 0.0)


35 1003 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.0) (0.28, 0.0)


## E9: a conserved economy of importance

32 units share 3200 integer tokens and a pool. Each tick each unit has input with probability 0.1 or 0.5, switching
every 2000 ticks. The focus is every unit with input whose holding is at least a threshold, which a controller moves by
the focus's size minus 4 each tick (clamped at zero). Each focused unit is *used* with probability 0.5. Four ways of
charging: *rent*, a twentieth of its holding (integer division) from each focused unit; a *proportional levy*, a
twentieth from every unit, in focus or not; a *flat levy*, one token from every unit holding one; and *none*. The pool
is paid out equally to the used units (integer division, remainder kept in the pool). Conservation is asserted every
tick. Two starts: *equal*, 100 tokens each; and *planted*, unit 0 holding half the total and the rest shared equally.
Printed per run: over the second half of 20 000 ticks, the largest share of all tokens any unit held and the share of
ticks whose 100-tick moving average of the focus's size is outside [3, 5]; and unit 0's mean share over the last 2000
ticks. A first version of the levy, a 400th of each holding, moved no token at all (every holding of 100 rounds it to
zero), and was dropped.

In [2]:
def e9(seed, charge, planted=False, n=32, total=3200, target=4, ticks=20000, window=100):
    rng = np.random.default_rng(seed)
    hold = np.full(n, total // n, dtype=np.int64)
    if planted:
        hold[:] = (total // 2) // (n - 1)
        hold[0] = total - hold[1:].sum()
    pool = theta = 0
    largest = 0.0
    sizes, first = [], []
    for t in range(ticks):
        p = 0.1 if (t // 2000) % 2 == 0 else 0.5
        focus = np.flatnonzero((rng.random(n) < p) & (hold >= theta))
        used = focus[rng.random(focus.size) < 0.5]
        if charge == "rent":
            c = hold[focus] // 20
            hold[focus] -= c
        elif charge == "proportional":
            c = hold // 20
            hold -= c
        elif charge == "flat":
            c = (hold > 0).astype(np.int64)
            hold -= c
        else:
            c = np.zeros(1, dtype=np.int64)
        pool += int(c.sum())
        if used.size:
            share = pool // used.size
            hold[used] += share
            pool -= share * used.size
        theta = max(0, theta + focus.size - target)
        assert hold.sum() + pool == total and (hold >= 0).all()
        sizes.append(focus.size)
        if t >= ticks // 2:
            largest = max(largest, hold.max() / total)
        if t >= ticks - 2000:
            first.append(hold[0] / total)
    s = np.array(sizes[ticks // 2:])
    avg = np.convolve(s, np.ones(window) / window, "valid")
    outside = float(np.mean((avg < target - 1) | (avg > target + 1)))
    return round(float(largest), 3), round(outside, 3), round(float(np.mean(first)), 3)

print("seed start | (largest share, outside band, unit 0's late share) for rent, proportional, flat, none")
for seed in (1001, 1002, 1003):
    for planted in (False, True):
        print(seed, "planted" if planted else "equal", "|",
              *[e9(seed, c, planted) for c in ("rent", "proportional", "flat", "none")])


seed start | (largest share, outside band, unit 0's late share) for rent, proportional, flat, none


1001 equal | (0.089, 0.043, 0.044) (0.318, 0.06, 0.007) (0.279, 0.193, 0.0) (0.031, 0.04, 0.031)


1001 planted | (0.099, 0.042, 0.035) (0.318, 0.06, 0.007) (0.477, 0.213, 0.335) (0.506, 0.04, 0.506)


1002 equal | (0.096, 0.035, 0.021) (0.411, 0.101, 0.015) (0.31, 0.207, 0.0) (0.031, 0.044, 0.031)


1002 planted | (0.098, 0.034, 0.021) (0.411, 0.101, 0.015) (0.268, 0.218, 0.165) (0.506, 0.041, 0.506)


1003 equal | (0.092, 0.054, 0.025) (0.371, 0.095, 0.04) (0.298, 0.186, 0.0) (0.031, 0.059, 0.031)


1003 planted | (0.107, 0.064, 0.027) (0.371, 0.095, 0.04) (0.321, 0.176, 0.173) (0.506, 0.052, 0.506)


## E10: input that reshapes the landscape

A Hopfield network of N = 200 sign units holding P = 10 random patterns by Hebb's rule (zero diagonal), updated one unit
at a time in index order (a unit keeps its state at zero input), settled when a sweep changes nothing, at most 50
sweeps. A cue takes each unit from pattern a with probability 0.6 and from pattern b otherwise (*blended*); in the
*shifting* condition the cue is redrawn before each of the first ten sweeps with that probability falling from 0.8 by
0.06 a sweep, and held after. The cue is also a persistent field of strength gamma. The baseline weighs every pattern
alike; the modulated network weighs pattern mu by the square of its positive overlap with the current cue,
recomputed each sweep, either as is or normalised so the weights sum to P. The *gain-matched* baseline weighs every
pattern alike at the normalised network's largest weight, recomputed each sweep, so that it differs from the
normalised network only in weighing the patterns unequally. Printed per run: the share of cues settling
clean (signed overlap at least 0.9 with one pattern), unsettled, and on a mirror (overlap at most -0.9).

In [3]:
N = 200

def e10(seed, gamma, mode, shift, P=10, cues=100):
    rng = np.random.default_rng(seed)
    X = rng.choice([-1.0, 1.0], size=(P, N))
    W0 = X.T @ X / N
    np.fill_diagonal(W0, 0)
    clean = unsettled = mirror = 0
    for _ in range(cues):
        a, b = rng.choice(P, 2, replace=False)
        u = np.where(rng.random(N) < 0.6, X[a], X[b])
        s = u.copy()
        settled = False
        for sweep in range(50):
            if shift and sweep < 10:
                u = np.where(rng.random(N) < 0.8 - 0.06 * sweep, X[a], X[b])
            w = np.maximum(X @ u / N, 0) ** 2
            if mode != "unnormalised":
                w = w * P / w.sum() if w.sum() > 0 else np.ones(P)
            if mode == "baseline":
                W = W0
            elif mode == "matched":
                W = W0 * w.max()
            else:
                W = (X.T * w) @ X / N
                np.fill_diagonal(W, 0)
            changed = 0
            for i in range(N):
                h = W[i] @ s + gamma * u[i]
                new = s[i] if h == 0 else (1.0 if h > 0 else -1.0)
                if new != s[i]:
                    s[i] = new; changed += 1
            if changed == 0 and not (shift and sweep < 10):
                settled = True
                break
        if not settled:
            unsettled += 1
            continue
        ov = X @ s / N
        if ov.max() >= 0.9:
            clean += 1
        elif ov.min() <= -0.9:
            mirror += 1
    return clean / cues, unsettled / cues, mirror / cues

print("condition gamma seed | baseline | unnormalised | normalised | gain-matched   (clean, unsettled, mirror)")
for shift in (False, True):
    for gamma in (0.0, 0.5):
        for seed in (1001, 1002, 1003):
            print("shifting" if shift else "blended", gamma, seed, "|",
                  *[e10(seed, gamma, m, shift) for m in ("baseline", "unnormalised", "normalised", "matched")])


condition gamma seed | baseline | unnormalised | normalised | gain-matched   (clean, unsettled, mirror)


blended 0.0 1001 | (0.94, 0.0, 0.01) (1.0, 0.0, 0.0) (1.0, 0.0, 0.0) (0.94, 0.0, 0.01)


blended 0.0 1002 | (0.87, 0.0, 0.0) (1.0, 0.0, 0.0) (1.0, 0.0, 0.0) (0.87, 0.0, 0.0)


blended 0.0 1003 | (0.92, 0.0, 0.0) (1.0, 0.0, 0.0) (1.0, 0.0, 0.0) (0.92, 0.0, 0.0)


blended 0.5 1001 | (0.4, 0.0, 0.0) (0.0, 0.0, 0.0) (0.99, 0.0, 0.0) (0.92, 0.0, 0.01)


...

shifting 0.0 1002 | (0.79, 0.0, 0.02) (0.85, 0.0, 0.15) (0.85, 0.0, 0.15) (0.79, 0.0, 0.02)


shifting 0.0 1003 | (0.94, 0.0, 0.01) (0.91, 0.0, 0.09) (0.91, 0.0, 0.09) (0.94, 0.0, 0.01)


shifting 0.5 1001 | (0.98, 0.0, 0.0) (0.12, 0.0, 0.0) (0.92, 0.0, 0.08) (0.94, 0.0, 0.0)


shifting 0.5 1002 | (0.86, 0.0, 0.0) (0.17, 0.0, 0.0) (0.95, 0.0, 0.05) (0.94, 0.0, 0.01)


shifting 0.5 1003 | (0.98, 0.0, 0.0) (0.16, 0.0, 0.0) (0.95, 0.0, 0.05) (0.98, 0.0, 0.0)


## What the probes informed

Read with the printed tables above, on these seeds only:

- **E6.** With the pair-swap dynamics no cue failed to settle. At `keep` = 25 the store without separation retrieved
  part of the similar cues and the store with separation nearly all; at 20 both retrieved all; at 30 the store without
  separation retrieved none and the one with separation most; at 35 both failed on most. The plan fixes 25 for its test
  and reports all four.
- **E9.** From a planted monopoly, unit 0's late share fell well below 0.25 under rent and under the proportional levy,
  and stayed at about half with no charge at all, so the recovery measure can fail. From an equal start the
  proportional levy, charged on every unit, let one unit gather far more than rent did. The share of ticks outside the
  band was close to the 0.05 margin under rent, and similar with no charge, so the controller alone sets it.
- **E10.** Normalised, the modulation settled a blended cue clean on nearly every cue; the gain-matched baseline did on
  most, the plain baseline on few at gamma = 0.5. In the shifting condition the gain-matched baseline was cleaner than
  the normalised modulation on two seeds of three and within 0.01 on the third, and the modulation settled on a mirror
  on some cues.